In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler
from jumpmodels.jump import JumpModel

print("all good")

In [ ]:
# pull daily adjusted close for all three indices
# the three tickers pulled are the s&p 500, nasdaq composite and DJIA. ^ denotes index symbols rather than individual stocks
tickers = ['^GSPC', '^IXIC', '^DJI']

# auto_adjust means that the prices are adjusted for stock splits & dividends for historical accuracy
# we just take the closing price of each day
raw = yf.download(tickers, start='1970-01-01', end='2026-05-26', auto_adjust=True)['Close']

# rename for clarity
raw.columns = ['SP500', 'Nasdaq', 'DJI']

# log returns
# raw.shift(1) - shifts prices 1 day forwards so that each row is today's price / yesterday's price
# np.log() - takes the natural log of the above ratio, giving log returns
# .dropna() removes the first row, which has no previous day to compute a return from
returns = np.log(raw / raw.shift(1)).dropna()

print(returns.shape)
print(returns.head(5))
print("\nmissing values:")
print(returns.isnull().sum())

In [ ]:
# sets the rolling window to 20 days, which is approx. 1 trading month
# why? paper replication
# ~ potentially in the future we can run a sensitivity analysis on this
window = 20

# empty dataframe with the same dates as the returns data.
# will be populated with the 2 feature columns
features = pd.DataFrame(index=returns.index)

# using SP500 as primary series - same as paper

# for each day, computes the daily average return over the past 20 days.
# captures whether the market has been trending up or down recently
features['rolling_return'] = returns['SP500'].rolling(window).mean()

# for each day, compute the STD of daily returns over the past 20 days.
# captures how much the market has been bouncing around
# typically calm periods show values like 0.005-0.008, turbulent periods are about 0.02-0.06
features['rolling_vol'] = returns['SP500'].rolling(window).std()

# first 19 rows cant have a rolling 20-day calculation so they are dropped
features = features.dropna()

print(features.shape)
print(features.head(5))

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ax1.plot(features.index, features['rolling_return'], color='steelblue', linewidth=0.8)
ax1.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax1.set_title('20-day Rolling Return (SP500)')
ax1.set_ylabel('Return')

ax2.plot(features.index, features['rolling_vol'], color='crimson', linewidth=0.8)
ax2.set_title('20-day Rolling Volatility (SP500)')
ax2.set_ylabel('Volatility')

plt.tight_layout()
plt.show()

'''
Rolling return:
- most days clustered around 0
- deep negative spikes correspond to major crashes, eg. dot-com, COVID, etc.
- returns are roughly symmetric, however negative extremes are slightly more extreme

Rolling vol:
- baseline is around 0.008-0.010 during calm periods - the "normal" market
- 2008: spikes to about 0.05
- 2020: spikes to about 0.065 (largest in 32-year history)
- 2022: spike is visible but much less dramatic (grinding bear market, not sudden crash)
'''

In [ ]:
# split dates
# everything before 2006 is training
# 2006-2014 is validation
# 2015 onwards is test
train_end = '2005-12-31'
val_end   = '2014-12-31'

# selects all rows where the date is on or before Dec 31 2005
X_train = features[features.index <= train_end]

# selects rows between Jan 2006 and Dec 2014 (sandbox for tuning lambda, includes 2008 financial crisis)
X_val   = features[(features.index > train_end) & (features.index <= val_end)]

# everything after Dec 2014 is used for testing
X_test  = features[features.index > val_end]

# note: each major stress event falls in a different period:
# dot-com in training, financial crisis in validation, COVID and 2022 in test
# this ensures that the model won't memorize a crash pattern and will be
# validated against new types of market stress it never encountered during development

print(f"Train: {X_train.shape[0]} days ({X_train.index[0].date()} to {X_train.index[-1].date()})")
print(f"Val:   {X_val.shape[0]} days ({X_val.index[0].date()} to {X_val.index[-1].date()})")
print(f"Test:  {X_test.shape[0]} days ({X_test.index[0].date()} to {X_test.index[-1].date()})")

## Hidden Markov Model

##### States
- State 0: calm market
- State 1: turbulent market

##### Emissions
- 20-day rolling returns
- 20-day rolling volatility

##### Process
The HMM is trained using the **Baum-Welch algorithm** (an EM algorithm) which learns:
- The Transition Probability Matrix (TPM)
- The Gaussian emission distributions for each hidden state
- The initial state probabilities

Once trained, the **Viterbi algorithm** decodes the most likely sequence of hidden states given the observations.

In [ ]:
# scale features — HMM is sensitive to feature magnitudes
# StandardScaler transforms each feature so it has mean=0 and std=1
# without this, the HMM would effectively ignore rolling_return (tiny values ~0.0003)
# relative to rolling_vol (larger values ~0.009)
scaler = StandardScaler()

# fit on training data only to prevent data leakage
# transform applies the learned normalization to each split
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# scale the full dataset using training scaler (for full-history visualization)
X_all_scaled = scaler.transform(features)

# create the HMM:
# n_components=2: two hidden states (calm, turbulent)
# covariance_type='full': each state gets its own covariance matrix
# n_iter=1000: run Baum-Welch up to 1000 iterations
# random_state=42: fixes random seed for reproducibility
model_hmm = hmm.GaussianHMM(
    n_components=2,
    covariance_type='full',
    n_iter=1000,
    random_state=42
)

# Baum-Welch trains on 1992-2005 data only
model_hmm.fit(X_train_scaled)

# Viterbi decodes the most likely regime sequence across the full history
hmm_regimes = model_hmm.predict(X_all_scaled)
features['hmm_regime'] = hmm_regimes

# get soft probabilities for each state (forward-backward algorithm)
probs_all = model_hmm.predict_proba(X_all_scaled)
features['hmm_p_state0'] = probs_all[:, 0]
features['hmm_p_state1'] = probs_all[:, 1]

print("State means (rolling_return, rolling_vol) — standardized:")
print(model_hmm.means_.round(4))
print("\nRegime counts:")
print(features['hmm_regime'].value_counts())
print("\nTransition matrix:")
print(model_hmm.transmat_.round(4))

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

sp500_price = raw['SP500'][features.index]
ax1.plot(features.index, sp500_price, color='steelblue', linewidth=0.8)
ax1.set_ylabel('SP500 Price')
ax1.set_title('SP500 Price with HMM Regimes')

turbulent = features['hmm_regime'] == 1
ax1.fill_between(features.index, sp500_price.min(), sp500_price.max(),
                 where=turbulent, alpha=0.3, color='red', label='Turbulent')
ax1.legend()

ax2.plot(features.index, features['hmm_p_state1'],
         color='crimson', linewidth=0.8, label='P(turbulent)')
ax2.axhline(0.5, color='black', linewidth=0.5, linestyle='--')
ax2.set_ylabel('P(turbulent)')
ax2.set_title('HMM Turbulent Regime Probability')
ax2.legend()

plt.tight_layout()
plt.show()

switches_hmm = (features['hmm_regime'].diff() != 0).sum()
print(f"Total regime switches: {switches_hmm}")
print(f"Average days between switches: {len(features) / switches_hmm:.1f}")

In [ ]:
# align returns with features index
strat_returns = returns['SP500'][features.index].copy()

# calm=state 0 → invest; turbulent=state 1 → cash
# .shift(1) avoids lookahead bias: yesterday's regime determines today's position
features['hmm_position'] = (features['hmm_regime'] == 0).astype(int).shift(1)

# strategy return = position × market return
# position=1: full market return; position=0: zero (cash)
features['hmm_strat_return'] = features['hmm_position'] * strat_returns

# buy and hold benchmark
features['bh_return'] = strat_returns

print("Strategy return sample:")
print(features[['hmm_position', 'hmm_strat_return', 'bh_return']].head(10))

In [ ]:
def evaluate_strategy(returns_series, label, rf=0.0):
    r = returns_series.dropna()
    total_return  = r.sum()
    ann_return    = r.mean() * 252          # annualize: 252 trading days
    ann_vol       = r.std() * np.sqrt(252)  # annualize: std scales with sqrt(time)
    sharpe        = (ann_return - rf) / ann_vol

    # max drawdown
    cumulative  = (1 + r).cumprod()
    rolling_max = cumulative.cummax()
    drawdown    = (cumulative - rolling_max) / rolling_max
    max_dd      = drawdown.min()

    print(f"\n{label}")
    print(f"  Annualized Return : {ann_return:.4f}")
    print(f"  Annualized Vol    : {ann_vol:.4f}")
    print(f"  Sharpe Ratio      : {sharpe:.4f}")
    print(f"  Max Drawdown      : {max_dd:.4f}")

# boolean mask for test period — only evaluate on unseen data
test_idx = features.index > val_end

## HMM Strategy: Buy and Hold vs HMM Binary

### Buy and Hold
- Invest on day 1 and never touch it again. Always 100% invested.

### HMM Binary
- Calm regime (state 0): 100% invested in S&P 500
- Turbulent regime (state 1): 100% in cash, earning nothing
- Transaction cost: 0.1% deducted every time the strategy switches position

In [ ]:
# transaction cost: 0.1% per switch
tc = 0.001

position = features['hmm_position'].fillna(0)
switches = (position.diff().abs() > 0).astype(int)

features['hmm_strat_return_tc'] = features['hmm_strat_return'] - switches * tc

print(f"Total switches (full period): {switches.sum()}")
print(f"Total switches (test period): {switches[test_idx].sum()}")

evaluate_strategy(features.loc[test_idx, 'bh_return'],           'Buy and Hold')
evaluate_strategy(features.loc[test_idx, 'hmm_strat_return'],    'HMM Binary (no TC)')
evaluate_strategy(features.loc[test_idx, 'hmm_strat_return_tc'], 'HMM Binary (with TC)')

## Analysis: HMM

### Finding
HMM regime switching underperforms buy-and-hold on a risk-adjusted basis once transaction costs are included.
The noisy, flickering regime sequence causes the strategy to sit in cash during many calm profitable days —
missing upside without a compensating crash-avoidance benefit.

This motivates the Statistical Jump Model, which adds an explicit switching penalty to force more persistent,
cleaner regimes.

In [ ]:
# prepare scaled DataFrames for jump model
# jumpmodels library requires a proper DataFrame with a datetime index (not raw numpy arrays)
X_train_jm = pd.DataFrame(
    scaler.transform(X_train),
    index=X_train.index,
    columns=X_train.columns
)

X_all_jm = pd.DataFrame(
    scaler.transform(features[['rolling_return', 'rolling_vol']]),
    index=features.index,
    columns=['rolling_return', 'rolling_vol']
)

# validation returns — used by both lambda sweeps
val_returns = strat_returns[(strat_returns.index > train_end) & (strat_returns.index <= val_end)]

print(f"X_train_jm shape: {X_train_jm.shape}")
print(f"X_all_jm shape:   {X_all_jm.shape}")
print(f"val_returns:      {val_returns.shape[0]} days")

In [ ]:
# ── Discrete Jump Model λ sweep on validation ────────────────────────────────
# tune jump_penalty for the discrete JM by maximising Sharpe on the validation set
lambda_values_discrete = [10, 25, 50, 100, 200, 300, 500, 800, 1000]

results_discrete = []

for lam in lambda_values_discrete:
    jm_lam_d = JumpModel(n_components=2, jump_penalty=lam, cont=False)
    jm_lam_d.fit(X_train_jm, sort_by="cumret")

    # predict on train+val block, then slice to val dates
    trainval_mask   = features.index <= val_end
    X_trainval_jm_d = pd.DataFrame(
        scaler.transform(features.loc[trainval_mask, ['rolling_return', 'rolling_vol']]),
        index=features.index[trainval_mask],
        columns=['rolling_return', 'rolling_vol']
    )

    regimes_trainval = jm_lam_d.predict(X_trainval_jm_d)
    regimes_val = pd.Series(
        regimes_trainval,
        index=features.index[trainval_mask]
    ).loc[val_returns.index]

    # binary position — calm=0 → invest
    position_val = (regimes_val == 0).astype(int).shift(1).fillna(0)
    strat_ret    = position_val * val_returns
    pos_changes  = position_val.diff().abs().fillna(0)
    strat_ret_tc = strat_ret - pos_changes * tc

    ann_ret = strat_ret_tc.mean() * 252
    ann_vol = strat_ret_tc.std() * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0

    results_discrete.append({'lambda': lam, 'sharpe': sharpe, 'ann_return': ann_ret})
    print(f"λ = {lam:6.0f}  |  Sharpe = {sharpe:.4f}  |  Ann. Return = {ann_ret:.4f}")

results_discrete_df  = pd.DataFrame(results_discrete)
best_row_discrete    = results_discrete_df.loc[results_discrete_df['sharpe'].idxmax()]
best_lambda_discrete = best_row_discrete['lambda']
print(f"\nBest λ (discrete JM) = {best_lambda_discrete} with validation Sharpe = {best_row_discrete['sharpe']:.4f}")

In [ ]:
# fit discrete jump model with tuned lambda
# cont=False: produces hard 0/1 regime labels (discrete version)
# sort_by='cumret': ensures state 0 = higher cumulative return = calm regime
jm = JumpModel(n_components=2, jump_penalty=best_lambda_discrete, cont=False)
jm.fit(X_train_jm, sort_by="cumret")

# predict hard regime labels across full history
jm_regimes = jm.predict(X_all_jm)
features['jm_regime'] = jm_regimes

# store probabilities (will be hard 0/1 for discrete model)
jm_probs = jm.predict_proba(X_all_jm)
features['jm_p_state0'] = jm_probs.iloc[:, 0]
features['jm_p_state1'] = jm_probs.iloc[:, 1]

print("Regime counts:")
print(features['jm_regime'].value_counts())
jm_switches = (features['jm_regime'].diff() != 0).sum()
print(f"\nTotal regime switches: {jm_switches}")
print(f"Average days between switches: {len(features) / jm_switches:.1f}")
print(f"\nHMM switches for comparison: {switches.sum()}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

sp500_price = raw['SP500'][features.index]
ax1.plot(features.index, sp500_price, color='steelblue', linewidth=0.8)
ax1.set_ylabel('SP500 Price')
ax1.set_title('SP500 Price with Jump Model Regimes')

turbulent_jm = features['jm_regime'] == 1
ax1.fill_between(features.index, sp500_price.min(), sp500_price.max(),
                 where=turbulent_jm, alpha=0.3, color='red', label='Turbulent')
ax1.legend()

ax2.plot(features.index, features['jm_p_state1'],
         color='crimson', linewidth=0.8, label='P(turbulent)')
ax2.axhline(0.5, color='black', linewidth=0.5, linestyle='--')
ax2.set_ylabel('P(turbulent)')
ax2.set_title('Jump Model Turbulent Regime Probability')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# discrete jump model binary strategy — same structure as HMM
# state 0 = calm → invest; state 1 = turbulent → cash
features['jm_position'] = (features['jm_regime'] == 0).astype(int).shift(1)
features['jm_strat_return'] = features['jm_position'] * strat_returns

# transaction costs
jm_position    = features['jm_position'].fillna(0)
jm_switches_ts = (jm_position.diff().abs() > 0).astype(int)
features['jm_strat_return_tc'] = features['jm_strat_return'] - jm_switches_ts * tc

print(f"JM switches (test period): {jm_switches_ts[test_idx].sum()}")
print(f"HMM switches (test period): {switches[test_idx].sum()}")

evaluate_strategy(features.loc[test_idx, 'bh_return'],          'Buy and Hold')
evaluate_strategy(features.loc[test_idx, 'hmm_strat_return_tc'],'HMM Binary (with TC)')
evaluate_strategy(features.loc[test_idx, 'jm_strat_return_tc'], 'Jump Model Binary (with TC)')

In [ ]:
# ── Continuous Jump Model λ sweep on validation ──────────────────────────────
# tune jump_penalty for the CJM by maximising Sharpe on the validation set
lambda_values = [10, 25, 50, 100, 200, 300, 500, 800, 1000]

results = []

for lam in lambda_values:
    jm_lam = JumpModel(n_components=2, jump_penalty=lam, cont=True)
    jm_lam.fit(X_train_jm, sort_by="cumret")

    # predict on train+val block, then slice to val dates
    trainval_mask = features.index <= val_end
    X_trainval_jm = pd.DataFrame(
        scaler.transform(features.loc[trainval_mask, ['rolling_return', 'rolling_vol']]),
        index=features.index[trainval_mask],
        columns=['rolling_return', 'rolling_vol']
    )

    probs      = jm_lam.predict_proba(X_trainval_jm)
    p_calm_val = probs.iloc[:, 0].loc[val_returns.index]

    position_val = p_calm_val.shift(1).fillna(0)
    strat_ret    = position_val * val_returns
    pos_changes  = position_val.diff().abs().fillna(0)
    strat_ret_tc = strat_ret - pos_changes * tc

    ann_ret = strat_ret_tc.mean() * 252
    ann_vol = strat_ret_tc.std() * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0

    results.append({'lambda': lam, 'sharpe': sharpe, 'ann_return': ann_ret})
    print(f"λ = {lam:6.0f}  |  Sharpe = {sharpe:.4f}  |  Ann. Return = {ann_ret:.4f}")

results_df  = pd.DataFrame(results)
best_row    = results_df.loc[results_df['sharpe'].idxmax()]
best_lambda = best_row['lambda']
print(f"\nBest λ (CJM) = {best_lambda} with validation Sharpe = {best_row['sharpe']:.4f}")

In [ ]:
# fit continuous jump model with tuned lambda
# cont=True: produces smooth regime probabilities between 0 and 1
jm_cont = JumpModel(n_components=2, jump_penalty=best_lambda, cont=True)
jm_cont.fit(X_train_jm, sort_by="cumret")

cjm_probs = jm_cont.predict_proba(X_all_jm)
features['cjm_p_state0'] = cjm_probs.iloc[:, 0]
features['cjm_p_state1'] = cjm_probs.iloc[:, 1]

print("CJM probability distribution:")
print(features['cjm_p_state0'].describe().round(4))
print("\nSample unique values (should include values between 0 and 1):")
print(features['cjm_p_state0'].round(2).value_counts().head(10))

In [ ]:
# continuous position = P(calm) — scales exposure proportionally to regime confidence
# position=0.8 → 80% invested; position=0.2 → 20% invested
features['cjm_position']    = features['cjm_p_state0'].shift(1)
features['cjm_cont_return'] = features['cjm_position'] * strat_returns

# proportional transaction costs — scales with how much the position actually moves
cjm_pos = features['cjm_position'].fillna(0)
cjm_position_changes = cjm_pos.diff().abs()
features['cjm_cont_return_tc'] = features['cjm_cont_return'] - cjm_position_changes * tc

evaluate_strategy(features.loc[test_idx, 'bh_return'],           'Buy and Hold')
evaluate_strategy(features.loc[test_idx, 'hmm_strat_return_tc'], 'HMM Binary (with TC)')
evaluate_strategy(features.loc[test_idx, 'jm_strat_return_tc'],  'Jump Model Binary (with TC)')
evaluate_strategy(features.loc[test_idx, 'cjm_cont_return_tc'],  'CJM Continuous (with TC)')

## Analysis: Jump Model vs HMM vs CJM

### Three-step progression

**HMM** — noisy regime switching (100+ switches) produces poor risk-adjusted returns. Frequent transitions to cash
miss too much upside without meaningfully avoiding downside. Sharpe worse than buy-and-hold after transaction costs.

**Discrete Jump Model** — explicit jump penalty forces regime persistence (fewer switches, longer durations).
Regime boundaries align cleanly with known market stress periods. Sharpe approximately doubles vs HMM.

**Continuous Jump Model** — replaces binary all-in/all-out with smooth position sizing proportional to P(calm).
Avoids abrupt switches, captures partial upside during uncertain transitions, reduces transaction drag.
Achieves highest Sharpe while exceeding buy-and-hold returns with a fraction of the drawdown.

### Key structural insight
The organic asymmetry of the continuous strategy: during regime transitions, position scales down gradually
rather than snapping to zero — capturing partial market exposure while reducing risk. Neither strategy
achieves this; only the CJM's smooth probabilities enable it.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

strategies = {
    'Buy and Hold':                'bh_return',
    'HMM Binary (with TC)':        'hmm_strat_return_tc',
    'Jump Model Binary (with TC)': 'jm_strat_return_tc',
    'CJM Continuous (with TC)':    'cjm_cont_return_tc',
}

colors = {
    'Buy and Hold':                'steelblue',
    'HMM Binary (with TC)':        'crimson',
    'Jump Model Binary (with TC)': 'darkorange',
    'CJM Continuous (with TC)':    'green',
}

test_features = features.loc[test_idx]

for label, col in strategies.items():
    cumret = (1 + test_features[col].fillna(0)).cumprod()
    ax.plot(test_features.index, cumret,
            label=label, color=colors[label], linewidth=1.5)

# shade CJM turbulent periods
turbulent_test = test_features['cjm_p_state0'].shift(1).fillna(1) < 0.5
ax.fill_between(test_features.index, 0, 4,
                where=turbulent_test, alpha=0.08, color='red', label='CJM Turbulent')

ax.set_title('Cumulative Returns — Test Period (2015–2026)', fontsize=14, fontweight='bold')
ax.set_ylabel('Portfolio Value ($1 invested)', fontsize=12)
ax.set_xlabel('Date', fontsize=12)
ax.legend(loc='upper left', fontsize=10)
ax.axhline(1.0, color='black', linewidth=0.5, linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

for label, col in strategies.items():
    cumret = (1 + test_features[col].fillna(0)).cumprod()
    final_val = cumret.iloc[-1]
    ax.annotate(f'${final_val:.2f}',
                xy=(test_features.index[-1], final_val),
                xytext=(8, 0), textcoords='offset points',
                fontsize=9, color=colors[label], fontweight='bold')

plt.tight_layout()
plt.show()

print("\nFinal portfolio value ($1 invested in 2015):")
for label, col in strategies.items():
    cumret = (1 + test_features[col].fillna(0)).cumprod()
    print(f"  {label}: ${cumret.iloc[-1]:.2f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LAYER 2 — SECTOR ROTATION
# Train:    1999–2005  (sector ETF data starts ~1999)
# Validate: 2006–2014
# Test:     2015–2026
# ═══════════════════════════════════════════════════════════════

# ── Cell 1: Pull sector data ─────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from jumpmodels.jump import JumpModel

sector_tickers = {
    'XLK': 'Technology',
    'XLF': 'Financials',
    'XLE': 'Energy',
    'XLV': 'Healthcare',
    'XLY': 'Consumer Discretionary',
    'XLP': 'Consumer Staples',
    'XLU': 'Utilities',
    'XLI': 'Industrials',
    'XLB': 'Materials'
}
sectors = list(sector_tickers.keys())

raw_sectors = yf.download(
    ['SPY'] + sectors,
    start='1998-01-01',
    end='2026-05-26',
    auto_adjust=True
)['Close']

sector_returns = np.log(raw_sectors / raw_sectors.shift(1)).dropna()

print(f"Shape: {sector_returns.shape}")
print(f"Date range: {sector_returns.index[0].date()} to {sector_returns.index[-1].date()}")
print(f"Missing values:\n{sector_returns.isnull().sum()}")

In [ ]:
# ── Cell 3: Date splits ───────────────────────────────────────
train_end    = '2005-12-31'
validate_end = '2014-12-31'
# test = 2015 onwards — DO NOT TOUCH until final evaluation

print(f"Train:    {sector_features['XLK'].index[0].date()} to {train_end}")
print(f"Validate: 2006-01-01 to {validate_end}")
print(f"Test:     2015-01-01 to {sector_features['XLK'].index[-1].date()}")

In [ ]:
# ── Cell 4: Fit scalers and jump models on train only ─────────
# scaler fit on train → applied to val and test (no leakage)
# jump model fit on train → predicts on val and test separately

sector_scalers = {}
sector_models  = {}
sector_regimes = {}

for ticker in sectors:
    feat_df = sector_features[ticker]

    train_mask = feat_df.index <= train_end
    val_mask   = (feat_df.index > train_end) & (feat_df.index <= validate_end)
    test_mask  = feat_df.index > validate_end

    # fit scaler on train only
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(feat_df.loc[train_mask, ['resid_return', 'resid_vol']])
    val_scaled   = scaler.transform(feat_df.loc[val_mask,       ['resid_return', 'resid_vol']])
    test_scaled  = scaler.transform(feat_df.loc[test_mask,      ['resid_return', 'resid_vol']])

    # fit model on train only
    jm = JumpModel(n_components=2, jump_penalty=50)  # penalty tuned in Cell 5
    jm.fit(train_scaled, sort_by="vol")

    # predict each split separately — no future data bleeds into past
    train_labels = jm.predict(train_scaled)
    val_labels   = jm.predict(val_scaled)
    test_labels  = jm.predict(test_scaled)

    labels = np.concatenate([train_labels, val_labels, test_labels])
    index  = feat_df.loc[train_mask | val_mask | test_mask].index

    sector_scalers[ticker] = scaler
    sector_models[ticker]  = jm
    sector_regimes[ticker] = pd.Series(labels, index=index, name=ticker)

print("Models fit. Calm % per sector (full history):")
for ticker in sectors:
    calm_pct = (sector_regimes[ticker] == 0).mean()
    print(f"  {ticker}: {calm_pct:.1%}")

In [ ]:
# ── Cell 5: Tune jump penalty on validate Sharpe only ─────────
# test set is never touched here

def sharpe(returns, periods=252):
    return (returns.mean() / returns.std()) * np.sqrt(periods)

def max_drawdown(cum_returns):
    roll_max = cum_returns.cummax()
    return ((cum_returns - roll_max) / roll_max).min()

def build_weights(regimes_dict, sectors):
    regime_df = pd.DataFrame(regimes_dict).dropna()
    regime_df['n_calm'] = (regime_df[sectors] == 0).sum(axis=1)
    w_df = pd.DataFrame(index=regime_df.index, columns=sectors, dtype=float)
    for date, row in regime_df.iterrows():
        n_calm = row['n_calm']
        if n_calm == 0:
            w_df.loc[date] = 0.0
        else:
            calm_mask = row[sectors] == 0
            w_df.loc[date] = 0.0
            w_df.loc[date, calm_mask] = 1 / n_calm
    return w_df

penalties   = [5, 10, 20, 30, 50, 75, 100, 150, 200, 300, 500]
best_sharpe = -np.inf
best_penalty = None
sweep_results = []

for penalty in penalties:
    temp_regimes = {}
    for ticker in sectors:
        feat_df    = sector_features[ticker]
        train_mask = feat_df.index <= train_end
        val_mask   = (feat_df.index > train_end) & (feat_df.index <= validate_end)
        test_mask  = feat_df.index > validate_end

        train_scaled = sector_scalers[ticker].transform(feat_df.loc[train_mask, ['resid_return', 'resid_vol']])
        val_scaled   = sector_scalers[ticker].transform(feat_df.loc[val_mask,   ['resid_return', 'resid_vol']])
        test_scaled  = sector_scalers[ticker].transform(feat_df.loc[test_mask,  ['resid_return', 'resid_vol']])

        jm = JumpModel(n_components=2, jump_penalty=penalty)
        jm.fit(train_scaled, sort_by="vol")

        labels = np.concatenate([
            jm.predict(train_scaled),
            jm.predict(val_scaled),
            jm.predict(test_scaled)
        ])
        index = feat_df.loc[train_mask | val_mask | test_mask].index
        temp_regimes[ticker] = pd.Series(labels, index=index, name=ticker)

    w_df      = build_weights(temp_regimes, sectors)
    strat_ret = (w_df * sector_returns[sectors].reindex(w_df.index)).sum(axis=1)
    val_ret   = strat_ret[(strat_ret.index > train_end) & (strat_ret.index <= validate_end)]
    val_sr    = sharpe(val_ret)
    sweep_results.append({'penalty': penalty, 'val_sharpe': val_sr})
    print(f"penalty={penalty:>4}  val_sharpe={val_sr:.4f}")

    if val_sr > best_sharpe:
        best_sharpe  = val_sr
        best_penalty = penalty

print(f"\nBest penalty: {best_penalty}  (val Sharpe: {best_sharpe:.4f})")

In [ ]:
# ── Cell 6: Refit final models with best penalty ──────────────
sector_regimes_final = {}

for ticker in sectors:
    feat_df    = sector_features[ticker]
    train_mask = feat_df.index <= train_end
    val_mask   = (feat_df.index > train_end) & (feat_df.index <= validate_end)
    test_mask  = feat_df.index > validate_end

    train_scaled = sector_scalers[ticker].transform(feat_df.loc[train_mask, ['resid_return', 'resid_vol']])
    val_scaled   = sector_scalers[ticker].transform(feat_df.loc[val_mask,   ['resid_return', 'resid_vol']])
    test_scaled  = sector_scalers[ticker].transform(feat_df.loc[test_mask,  ['resid_return', 'resid_vol']])

    jm = JumpModel(n_components=2, jump_penalty=best_penalty)
    jm.fit(train_scaled, sort_by="vol")

    labels = np.concatenate([
        jm.predict(train_scaled),
        jm.predict(val_scaled),
        jm.predict(test_scaled)
    ])
    index = feat_df.loc[train_mask | val_mask | test_mask].index
    sector_regimes_final[ticker] = pd.Series(labels, index=index, name=ticker)

weights_df = build_weights(sector_regimes_final, sectors)
print(f"Weights built. Shape: {weights_df.shape}")
print(f"Cash days (all turbulent): {(weights_df.sum(axis=1) == 0).sum()}")

In [ ]:
# ── Cell 7: Final evaluation ──────────────────────────────────
# test numbers are the honest out-of-sample result

strategy_returns = (weights_df * sector_returns[sectors].reindex(weights_df.index)).sum(axis=1)
spy_ret          = sector_returns['SPY'].reindex(weights_df.index)

splits = {
    'Train':    strategy_returns.index <= train_end,
    'Validate': (strategy_returns.index > train_end) & (strategy_returns.index <= validate_end),
    'Test':     strategy_returns.index > validate_end,
}

print(f"{'Period':<12} {'Strat Ret':>10} {'Strat Vol':>10} {'Strat SR':>10} {'Strat MDD':>10} | {'SPY Ret':>8} {'SPY Vol':>8} {'SPY SR':>8} {'SPY MDD':>10}")
print("-" * 100)
for period, mask in splits.items():
    s = strategy_returns[mask]
    b = spy_ret[mask]
    print(f"{period:<12} {s.mean()*252:>10.2%} {s.std()*np.sqrt(252):>10.2%} {sharpe(s):>10.3f} {max_drawdown((1+s).cumprod()):>10.2%} | {b.mean()*252:>8.2%} {b.std()*np.sqrt(252):>8.2%} {sharpe(b):>8.3f} {max_drawdown((1+b).cumprod()):>10.2%}")